# Session 9 · Search a support handbook by meaning
Build a support assistant in four steps: **Load → Split → Store → Ask**, turning one fictional company's Word handbook into searchable passages and cited answers. Gemini creates both the embeddings and the answers, while Pinecone stores and searches the vectors. Before running top to bottom, install `requirements.txt`, create `.env` from `.env.example`, add both keys, and select the Python 3.13 environment as your notebook kernel.

## 1 · Setup
Import the local document tools and set the two Gemini model names in one place. Paths start at the repository folder, so launch Jupyter there.

In [ ]:
import csv
import os
import time
from pathlib import Path
import numpy as np
from docx import Document
from docx.table import Table
from dotenv import load_dotenv

EMBEDDING_MODEL = "google_genai:gemini-embedding-001"
CHAT_MODEL = "google_genai:gemini-3.7-flash"
HANDBOOK_PATH = Path("data/support_handbook.docx")
INDEX_NAME = "trailmix-handbook"
K = 3

Load your keys from `.env` without displaying their values. If either key is missing, add it to that file and rerun this cell before continuing.

In [ ]:
load_dotenv(Path(".env"))
missing = [key for key in ("GOOGLE_API_KEY", "PINECONE_API_KEY")
           if not os.getenv(key)]
if missing:
    print("Missing " + ", ".join(missing) + ". Add them to .env, then rerun.")
else:
    print("Both keys are present. Ready to build the lab!")

These LangChain components are the same building blocks you used in Session 8. The splitter adds heading metadata, and the Pinecone integration gives us a retriever to connect with the `|` pipe.

In [ ]:
from langchain.embeddings import init_embeddings
from langchain.chat_models import init_chat_model
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_pinecone import PineconeVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from pinecone import Pinecone, ServerlessSpec

## 2 · Load the Word document
Walk paragraphs and tables in their original reading order with `iter_inner_content()`. Word heading styles become markdown headings, and each table row becomes a line with cells separated by ` | `.

In [ ]:
def docx_to_markdown(path: str | Path) -> str:
    """Convert body paragraphs and tables to markdown in reading order."""
    document = Document(path)
    blocks = []
    for item in document.iter_inner_content():
        if isinstance(item, Table):
            # Preserve each table's position, with one readable line per row.
            blocks.append("\n".join(" | ".join(c.text for c in r.cells) for r in item.rows))
        elif item.text.strip():
            # Word styles become the headings that will supply citations.
            prefix = {"Heading 1": "# ", "Heading 2": "## "}.get(item.style.name, "")
            blocks.append(prefix + item.text.strip())
    return "\n\n".join(blocks)

Now load the handbook and inspect the amount and shape of its text. **17 pages is too long to read for every customer question — this is why we search.**

In [ ]:
document = Document(HANDBOOK_PATH)
markdown = docx_to_markdown(HANDBOOK_PATH)
print(f"Paragraphs: {len(document.paragraphs)}")
print(f"Tables: {len(document.tables)}")
print(f"Total characters: {len(markdown):,}")
print("\nFirst 600 characters:\n")
print(markdown[:600])

## 3 · What is an embedding?
An embedding is a list of numbers that represents text so related meanings can be compared. This cell makes the first Gemini API call and shows the vector's size and first five values; it needs your Google key and may use API quota.

In [ ]:
emb = init_embeddings(EMBEDDING_MODEL)
vector = emb.embed_query("My tent let water in during a storm.")
print(f"Vector length: {len(vector)}")
print("First 5 numbers:", [round(value, 6) for value in vector[:5]])

## 4 · Similarity by meaning
Cosine similarity measures how closely two vectors point in the same direction. For this comparison we embed all three texts as documents so they use the same task settings; the rain-fly sentence should score higher even with few shared words.

In [ ]:
def cosine(a, b):
    a, b = np.asarray(a), np.asarray(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

sentences = [
    "my tent let water in",
    "The rain fly has a 3,000 mm waterproof rating.",
    "Trail Socks are merino-wool hiking socks sold in packs of three.",
]
vectors = emb.embed_documents(sentences)
for sentence, candidate in zip(sentences[1:], vectors[1:]):
    print(f"{cosine(vectors[0], candidate):.3f}  {sentence}")

## 5 · Split by heading
Split at section and topic headings instead of arbitrary character boundaries. Each piece keeps the heading names as metadata, giving us real citation labels such as `[Support teams > Safety escalations]`.

In [ ]:
splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[("#", "section"), ("##", "topic")]
)
pieces = splitter.split_text(markdown)
print(f"Pieces: {len(pieces)}")
print("Expected source outline: 7 sections and 58 topics.")

A citation is just a readable label built from a piece's metadata. The title page precedes every heading, so its label is `[Front matter]`.

In [ ]:
def citation_label(piece):
    section = piece.metadata.get("section", "Front matter")
    topic = piece.metadata.get("topic")
    return f"[{section} > {topic}]" if topic else f"[{section}]"

for piece in pieces[:15]:
    print(f"{citation_label(piece)} — {len(piece.page_content):,} characters")

## 6 · Create the Pinecone index
Ask Gemini for a vector and measure its dimension, because an index must match the vectors stored in it. Create the serverless cosine index only if it is missing; an existing index is reused.

In [ ]:
dimension = len(emb.embed_query("test"))
print(f"Embedding dimension: {dimension}")
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
if not pc.has_index(INDEX_NAME):
    pc.create_index(
        name=INDEX_NAME, dimension=dimension, metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    print(f"Created {INDEX_NAME}.")
else:
    print(f"Reusing {INDEX_NAME}.")

Index creation is asynchronous, so wait until Pinecone reports it ready. Also check the dimension and metric before storing anything, with a two-minute timeout to avoid waiting forever.

In [ ]:
deadline = time.monotonic() + 120
while not (description := pc.describe_index(INDEX_NAME)).status["ready"]:
    if time.monotonic() >= deadline:
        raise TimeoutError("Index is not ready; wait and retry this cell.")
    time.sleep(1)
if description.dimension != dimension or description.metric != "cosine":
    raise ValueError("Index mismatch: delete and recreate the lab index, then rerun.")
index = pc.Index(host=description.host)
print("Index ready.")

## 7 · Store
Upsert the pieces with stable IDs so rerunning the lab overwrites the same records. Pinecone needs a few seconds after a write before records become searchable, so pause before searching.

In [ ]:
store = PineconeVectorStore(index=index, embedding=emb)
ids = [f"piece-{i}" for i in range(len(pieces))]
store.add_documents(pieces, ids=ids)
time.sleep(5)
print(f"Stored {len(pieces)} pieces with stable IDs.")

## 8 · Search
Retrieve the three pieces whose vectors best match the gas-smell question. With a cosine index, higher scores mean greater similarity; inspect the labels and previews to see whether the safety topic appears.

In [ ]:
results = store.similarity_search_with_score(
    "customer smells gas from their stove", k=K
)
for piece, score in results:
    preview = piece.page_content[:120].replace("\n", " ")
    print(f"{citation_label(piece)} | score {score:.3f}\n{preview}\n")

## 9 · Evaluate retrieval
The five questions use different wording from the handbook to test search by meaning. Count a hit only when one of the top three pieces has both the expected section and topic; the target is at least four hits out of five.

In [ ]:
with Path("data/eval_questions.csv").open(newline="", encoding="utf-8") as file:
    questions = list(csv.DictReader(file))
hits = 0
for row in questions:
    found = store.similarity_search(row["question"], k=K)
    hit = any(p.metadata.get("section") == row["expected_section"]
              and p.metadata.get("topic") == row["expected_topic"] for p in found)
    hits += int(hit)
    print(f"{'✓' if hit else '✗'} {row['question']}")
    print(f"  Expected: [{row['expected_section']} > {row['expected_topic']}]")
    print("  Found:", ", ".join(citation_label(p) for p in found))
print(f"\nHit rate @3: {hits}/{len(questions)}")

## 10 · Ask with citations
Prefix every retrieved passage with the citation label we want in the answer. The retriever supplies only three passages per question, keeping the model's context focused on the relevant handbook text.

In [ ]:
def format_docs(pieces):
    return "\n\n".join(f"{citation_label(p)}\n{p.page_content}" for p in pieces)

retriever = store.as_retriever(search_kwargs={"k": K})

Give Gemini a system instruction to answer only from the excerpts and cite every claim. The human message contains the retrieved text followed by the customer's question, and an uncovered question should route to General Support.

In [ ]:
system_message = (
    "You are a TrailMix support assistant. Answer only from the handbook excerpts below. "
    "After each claim, cite the section in square brackets exactly as shown, "
    "e.g. [Policies > Returns]. If the excerpts do not answer the question, say: "
    "'The handbook doesn't cover this — route to General Support.'"
)
prompt = ChatPromptTemplate.from_messages([
    ("system", system_message),
    ("human", "Handbook excerpts:\n{excerpts}\n\nQuestion: {question}"),
])
model = init_chat_model(CHAT_MODEL)

Connect the steps with Session 8's `|` syntax. The dictionary runs the retrieval branch alongside the original question, and `StrOutputParser` turns the model response into text.

In [ ]:
chain = (
    {"excerpts": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt | model | StrOutputParser()
)

Try two handbook questions and one off-topic question. Check that every factual claim cites a real retrieved section and that the CEO question gets “The handbook doesn't cover this — route to General Support.”

In [ ]:
demo_questions = [
    "Customer says they can smell gas from their stove. What do I do?",
    "Can someone send back boots they already wore on one hike?",
    "What is the CEO's favourite hiking trail?",
]
for question in demo_questions:
    print(f"Question: {question}")
    print(f"Answer: {chain.invoke(question)}\n")

## 11 · Your turn
Edit `question` below and run the cell to ask your own question. Try “Can you ship my order to Japan?”, “What do I get with TrailMix Club?”, or “My Beacon 400 headlamp will not turn on; what should I try?”.

In [ ]:
question = "Can you ship my order to Japan?"
print(chain.invoke(question))